# HW3

วัชรมัย จันทวาลย์  B6722210

In [1]:
from collections import deque
from dataclasses import dataclass
from heapq import heappop, heappush
from itertools import count

START, GOAL = 'Arad', 'Vaslui'
edges = [
    ('Arad', 'Zerind', 75), ('Arad', 'Sibiu', 140), ('Arad', 'Timisoara', 118),
    ('Zerind', 'Oradea', 71), ('Oradea', 'Sibiu', 151),
    ('Timisoara', 'Lugoj', 111), ('Lugoj', 'Mehadia', 70),
    ('Mehadia', 'Drobeta', 75), ('Drobeta', 'Craiova', 120),
    ('Craiova', 'Rimnicu Vilcea', 146), ('Craiova', 'Pitesti', 138),
    ('Rimnicu Vilcea', 'Sibiu', 80), ('Rimnicu Vilcea', 'Pitesti', 97),
    ('Sibiu', 'Fagaras', 99), ('Fagaras', 'Bucharest', 211),
    ('Pitesti', 'Bucharest', 101), ('Bucharest', 'Giurgiu', 90),
    ('Bucharest', 'Urziceni', 85), ('Urziceni', 'Vaslui', 142),
    ('Vaslui', 'Iasi', 92), ('Iasi', 'Neamt', 87),
    ('Urziceni', 'Hirsova', 98), ('Hirsova', 'Eforie', 86),
]
graph = {}
for a, b, cost in edges:
    graph.setdefault(a, []).append((b, cost))
    graph.setdefault(b, []).append((a, cost))

def dijkstra_distances(goal):
    dist, frontier = {goal: 0}, [(0, goal)]
    while frontier:
        d, state = heappop(frontier)
        if d != dist[state]:
            continue
        for nxt, step in graph[state]:
            candidate = d + step
            if candidate < dist.get(nxt, float('inf')):
                dist[nxt] = candidate
                heappush(frontier, (candidate, nxt))
    return dist

h = dijkstra_distances(GOAL)
h


{'Vaslui': 0,
 'Urziceni': 142,
 'Iasi': 92,
 'Neamt': 179,
 'Bucharest': 227,
 'Hirsova': 240,
 'Fagaras': 438,
 'Pitesti': 328,
 'Giurgiu': 317,
 'Eforie': 326,
 'Craiova': 466,
 'Rimnicu Vilcea': 425,
 'Sibiu': 505,
 'Drobeta': 586,
 'Arad': 645,
 'Oradea': 656,
 'Mehadia': 661,
 'Zerind': 720,
 'Timisoara': 763,
 'Lugoj': 731}

In [2]:
@dataclass
class Result:
    algorithm: str
    path: list | None
    cost: int | None
    expanded: int
    generated: int
    depth: int | None

def result(name, path, cost, expanded, generated):
    return Result(name, path, cost, expanded, generated, None if path is None else len(path) - 1)

def bfs_tree(start, goal):
    q, expanded, generated = deque([(start, [start], 0)]), 0, 1
    while q:
        state, path, cost = q.popleft(); expanded += 1
        if state == goal: return result('Breadth-First Tree Search', path, cost, expanded, generated)
        for nxt, step in graph[state]:
            if nxt not in path:
                q.append((nxt, path + [nxt], cost + step)); generated += 1

def dfs_tree(start, goal):
    stack, expanded, generated = [(start, [start], 0)], 0, 1
    while stack:
        state, path, cost = stack.pop(); expanded += 1
        if state == goal: return result('Depth-First Tree Search', path, cost, expanded, generated)
        for nxt, step in reversed(graph[state]):
            if nxt not in path:
                stack.append((nxt, path + [nxt], cost + step)); generated += 1

def bfs_graph(start, goal):
    q, seen, expanded, generated = deque([(start, [start], 0)]), {start}, 0, 1
    while q:
        state, path, cost = q.popleft(); expanded += 1
        if state == goal: return result('Breadth-First Search', path, cost, expanded, generated)
        for nxt, step in graph[state]:
            if nxt not in seen:
                seen.add(nxt); q.append((nxt, path + [nxt], cost + step)); generated += 1

def dfs_graph(start, goal):
    stack, seen, expanded, generated = [(start, [start], 0)], set(), 0, 1
    while stack:
        state, path, cost = stack.pop()
        if state in seen: continue
        seen.add(state); expanded += 1
        if state == goal: return result('Depth-First Graph Search', path, cost, expanded, generated)
        for nxt, step in reversed(graph[state]):
            if nxt not in seen:
                stack.append((nxt, path + [nxt], cost + step)); generated += 1

def best_first_graph(start, goal, score, name):
    serial = count(); frontier = [(score(start, 0), next(serial), start, [start], 0)]
    best_g, expanded, generated = {start: 0}, 0, 1
    while frontier:
        _, _, state, path, cost = heappop(frontier)
        if cost != best_g.get(state): continue
        expanded += 1
        if state == goal: return result(name, path, cost, expanded, generated)
        for nxt, step in graph[state]:
            new_cost = cost + step
            if new_cost < best_g.get(nxt, float('inf')):
                best_g[nxt] = new_cost
                heappush(frontier, (score(nxt, new_cost), next(serial), nxt, path + [nxt], new_cost)); generated += 1

def depth_limited(start, goal, limit):
    expanded = generated = 0
    def visit(state, path, cost):
        nonlocal expanded, generated
        expanded += 1
        if state == goal: return path, cost
        if len(path) - 1 == limit: return None
        for nxt, step in graph[state]:
            if nxt not in path:
                generated += 1
                found = visit(nxt, path + [nxt], cost + step)
                if found: return found
        return None
    generated = 1; found = visit(start, [start], 0)
    return result(f'Depth-Limited Search (limit={limit})', *(found or (None, None)), expanded, generated)

def iterative_deepening(start, goal, max_depth=30):
    total_expanded = total_generated = 0
    for limit in range(max_depth + 1):
        r = depth_limited(start, goal, limit)
        total_expanded += r.expanded; total_generated += r.generated
        if r.path:
            return result('Iterative Deepening Search', r.path, r.cost, total_expanded, total_generated)

def rbfs(start, goal):
    expanded = generated = 0
    def visit(state, path, cost, f_limit):
        nonlocal expanded, generated
        expanded += 1
        if state == goal: return path, cost, cost
        successors = []
        for nxt, step in graph[state]:
            if nxt not in path:
                new_cost = cost + step
                successors.append([max(new_cost + h[nxt], cost + h[state]), nxt, new_cost]); generated += 1
        if not successors: return None, None, float('inf')
        while True:
            successors.sort(key=lambda x: x[0])
            best = successors[0]
            if best[0] > f_limit: return None, None, best[0]
            alternative = successors[1][0] if len(successors) > 1 else float('inf')
            found_path, found_cost, best[0] = visit(best[1], path + [best[1]], best[2], min(f_limit, alternative))
            if found_path: return found_path, found_cost, best[0]
    generated = 1; path, cost, _ = visit(start, [start], 0, float('inf'))
    return result('Recursive Best-First Search', path, cost, expanded, generated)


In [3]:
@dataclass
class GraphProblem:
    initial: str
    goal: str
    graph: dict
    heuristic: dict

    def goal_test(self, state):
        return state == self.goal

romania_problem = GraphProblem(START, GOAL, graph, h)

def breadth_first_tree_search(problem):
    return bfs_tree(problem.initial, problem.goal)

def depth_first_tree_search(problem):
    return dfs_tree(problem.initial, problem.goal)

def breadth_first_search_graph(problem):
    return bfs_graph(problem.initial, problem.goal)

def depth_first_graph_search(problem):
    return dfs_graph(problem.initial, problem.goal)

def best_first_graph_search(problem):
    return best_first_graph(problem.initial, problem.goal, lambda n, g: g + problem.heuristic[n], 'Best-First Graph Search (f=g+h)')

def uniform_cost_search_graph(problem):
    return best_first_graph(problem.initial, problem.goal, lambda n, g: g, 'Uniform-Cost Search')

def depth_limited_search_graph(problem, limit=5):
    return depth_limited(problem.initial, problem.goal, limit)

def iterative_deepening_search(problem):
    return iterative_deepening(problem.initial, problem.goal)

def greedy_best_first_search(problem):
    return best_first_graph(problem.initial, problem.goal, lambda n, g: problem.heuristic[n], 'Greedy Best-First Search')

def astar_search_graph(problem):
    return best_first_graph(problem.initial, problem.goal, lambda n, g: g + problem.heuristic[n], 'A* Search')

def recursive_best_first_search(problem):
    return rbfs(problem.initial, problem.goal)


In [4]:
LIMIT = 5
results = [
    breadth_first_tree_search(romania_problem),
    depth_first_tree_search(romania_problem),
    breadth_first_search_graph(romania_problem),
    depth_first_graph_search(romania_problem),
    best_first_graph_search(romania_problem),
    uniform_cost_search_graph(romania_problem),
    depth_limited_search_graph(romania_problem, LIMIT),
    iterative_deepening_search(romania_problem),
    greedy_best_first_search(romania_problem),
    astar_search_graph(romania_problem),
    recursive_best_first_search(romania_problem),
]

for r in results:
    print(f'{r.algorithm}:')
    print(f'  path = {" → ".join(r.path) if r.path else "ไม่พบคำตอบ"}')
    print(f'  cost = {r.cost}, depth = {r.depth}, expanded = {r.expanded}, generated = {r.generated}')


Breadth-First Tree Search:
  path = Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
  cost = 677, depth = 5, expanded = 37, generated = 56
Depth-First Tree Search:
  path = Arad → Zerind → Oradea → Sibiu → Rimnicu Vilcea → Craiova → Pitesti → Bucharest → Urziceni → Vaslui
  cost = 989, depth = 9, expanded = 16, generated = 21
Breadth-First Search:
  path = Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui
  cost = 677, depth = 5, expanded = 16, generated = 17
Depth-First Graph Search:
  path = Arad → Zerind → Oradea → Sibiu → Rimnicu Vilcea → Craiova → Pitesti → Bucharest → Urziceni → Vaslui
  cost = 989, depth = 9, expanded = 16, generated = 21
Best-First Graph Search (f=g+h):
  path = Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui
  cost = 645, depth = 6, expanded = 7, generated = 14
Uniform-Cost Search:
  path = Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui
  cost = 645, depth = 6, expanded = 17, generated = 19
Depth-Lim

In [5]:
import pandas as pd
summary = pd.DataFrame([r.__dict__ for r in results])
summary['path'] = summary['path'].map(lambda p: ' → '.join(p) if p else 'ไม่พบคำตอบ')
summary[['algorithm', 'path', 'cost', 'depth', 'expanded', 'generated']]


,algorithm,path,cost,depth,expanded,generated
0,Breadth-First Tree Search,Arad → Sibiu → Fagaras → Bucharest → Urziceni ...,677,5,37,56
1,Depth-First Tree Search,Arad → Zerind → Oradea → Sibiu → Rimnicu Vilce...,989,9,16,21
2,Breadth-First Search,Arad → Sibiu → Fagaras → Bucharest → Urziceni ...,677,5,16,17
3,Depth-First Graph Search,Arad → Zerind → Oradea → Sibiu → Rimnicu Vilce...,989,9,16,21
4,Best-First Graph Search (f=g+h),Arad → Sibiu → Rimnicu Vilcea → Pitesti → Buch...,645,6,7,14
5,Uniform-Cost Search,Arad → Sibiu → Rimnicu Vilcea → Pitesti → Buch...,645,6,17,19
6,Depth-Limited Search (limit=5),Arad → Sibiu → Fagaras → Bucharest → Urziceni ...,677,5,33,33
7,Iterative Deepening Search,Arad → Sibiu → Fagaras → Bucharest → Urziceni ...,677,5,87,87
8,Greedy Best-First Search,Arad → Sibiu → Rimnicu Vilcea → Pitesti → Buch...,645,6,7,14
9,A* Search,Arad → Sibiu → Rimnicu Vilcea → Pitesti → Buch...,645,6,7,14


## สรุปผลการค้นหา

1. **Breadth-First Tree Search**: ได้เส้นทาง Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 677, ความลึก 5. ค้นหาระดับตื้นที่สุด แต่มีเส้นทางซ้ำใน frontier ได้

2. **Depth-First Tree Search**: ได้เส้นทาง Arad → Zerind → Oradea → Sibiu → Rimnicu Vilcea → Craiova → Pitesti → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 989, ความลึก 9. เลือกค้นหาลึกก่อน จึงได้เส้นทางที่ยาวกว่า

3. **Breadth-First Search**: ได้เส้นทาง Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 677, ความลึก 5. เป็นคำตอบที่มีจำนวนขอบน้อยที่สุด

4. **Depth-First Graph Search**: ได้เส้นทางเดียวกับ Depth-First Tree Search, ค่าใช้จ่าย 989, ความลึก 9. มีการเก็บ explored set เพื่อลดการวนซ้ำ

5. **Best-First Graph Search**: ได้เส้นทาง Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 645, ความลึก 6. เลือกโหนดที่มีค่า f(n)=g(n)+h(n) ต่ำก่อน

6. **Uniform-Cost Search**: ได้เส้นทาง Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 645, ความลึก 6. เลือกต้นทุนสะสมต่ำสุดและได้คำตอบต้นทุนต่ำสุด

7. **Depth-Limited Search**: เมื่อกำหนด limit = 5 ได้เส้นทาง Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 677. ค้นหาได้พอดีที่ระดับจำกัด

8. **Iterative Deepening Search**: ได้เส้นทาง Arad → Sibiu → Fagaras → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 677, ความลึก 5. เพิ่ม depth limit ทีละระดับจนพบคำตอบ

9. **Greedy Best-First Search**: ได้เส้นทาง Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 645, ความลึก 6. ใช้ค่า h(n) เลือกโหนด และในกราฟนี้ได้คำตอบต้นทุนต่ำสุด

10. **A* Search**: ได้เส้นทาง Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 645, ความลึก 6. ใช้ค่า f(n)=g(n)+h(n) และได้คำตอบต้นทุนต่ำสุด

11. **Recursive Best-First Search**: ได้เส้นทาง Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui, ค่าใช้จ่าย 645, ความลึก 6. ใช้หลักการของ A* แต่ใช้หน่วยความจำน้อยกว่า

ดังนั้น เส้นทางที่มีต้นทุนต่ำที่สุดคือ Arad → Sibiu → Rimnicu Vilcea → Pitesti → Bucharest → Urziceni → Vaslui มีค่าใช้จ่ายรวม 645